# 05 — Model Training

Train LightGBM MultiOutputRegressor (24 independent models, one per horizon).
Also train quantile models for uncertainty bands.
Visualize predictions vs actuals for a sample week.

**Prerequisites:** Gold ABT built (NB03).

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

from ml.train import FEATURE_COLS, TARGET_COLS, train_lgbm_multioutput, train_quantile_models

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

con = duckdb.connect()
ABT_TRAIN = '../data/gold/abt_train.parquet'
ABT_TEST  = '../data/gold/abt_test.parquet'

## 5.1 Load training data

In [ ]:
train_df = con.execute(f'SELECT * FROM read_parquet("{ABT_TRAIN}")').fetchdf()
print(f'Train: {len(train_df):,} rows')
print(f'Features: {len(FEATURE_COLS)} | Targets: {len(TARGET_COLS)}')

# Sanity: no NaN in targets
nan_targets = train_df[TARGET_COLS].isna().sum().sum()
print(f'NaN in targets: {nan_targets} (should be 0)')

## 5.2 Train mean model

In [ ]:
print('Training mean model...')
model = train_lgbm_multioutput(train_df)
joblib.dump(model, MODELS_DIR / 'lgbm_multioutput.pkl')
print('Done.')

## 5.3 Train quantile models (10th, 90th percentile)

In [ ]:
print('Training quantile models...')
q_models = train_quantile_models(train_df, quantiles=(0.1, 0.9))
for q, m in q_models.items():
    q_str = f'q{int(q*100):02d}'
    joblib.dump(m, MODELS_DIR / f'lgbm_{q_str}.pkl')
    print(f'  q={q} saved')
print('Done.')

## 5.4 In-sample prediction check (training set)

In [ ]:
X_train = train_df[FEATURE_COLS]
preds = model.predict(X_train)
actuals = train_df[TARGET_COLS].values

mae_in_sample = np.abs(actuals - preds).mean(axis=0)
print('In-sample MAE per horizon (first 5, last 5):')
for i in list(range(5)) + list(range(19, 24)):
    print(f'  t+{i+1:02d}h: {mae_in_sample[i]:.3f} EUR/MWh')

## 5.5 Prediction vs actuals — sample week (test set)

In [ ]:
test_df = con.execute(f"SELECT * FROM read_parquet('{ABT_TEST}') LIMIT 200").fetchdf()

X_test = test_df[FEATURE_COLS]
preds_test = model.predict(X_test)
lower_test = q_models[0.1].predict(X_test)
upper_test = q_models[0.9].predict(X_test)

# Plot forecast horizon t+12h for the first 200 prediction moments
horizon_idx = 11  # t+12h
actual_vals  = test_df[f'price_t_plus_{horizon_idx+1}h'].values
pred_vals    = preds_test[:, horizon_idx]
lower_vals   = lower_test[:, horizon_idx]
upper_vals   = upper_test[:, horizon_idx]

ts = pd.to_datetime(test_df['timestamp_utc'], utc=True) + pd.Timedelta(hours=horizon_idx+1)

fig = go.Figure()
fig.add_traces([
    go.Scatter(x=list(ts)+list(ts[::-1]),
               y=list(upper_vals)+list(lower_vals[::-1]),
               fill='toself', fillcolor='rgba(99,110,250,0.15)',
               line=dict(color='rgba(0,0,0,0)'), name='80% interval'),
    go.Scatter(x=ts, y=pred_vals, name='Forecast', line=dict(color='#636EFA')),
    go.Scatter(x=ts, y=actual_vals, name='Actual', line=dict(color='#EF553B', dash='dot')),
])
fig.update_layout(title=f'Forecast vs Actual — t+12h (first 200 rows of OOT test)',
                  xaxis_title='', yaxis_title='EUR/MWh',
                  template='plotly_dark', height=400)
fig.show()